In [ ]:
import os
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

file_path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "data", "processed_dataset.csv")

df = pd.read_csv(file_path)

df.head()

### Création de variables utiles

In [16]:
cols_confort = ['Salon_Bin', 'SalleDeBainInterieure_Bin', 'Parking_Bin', 'Meuble_Bin', 'Jardin_Bin']
df['Indicateur_Confort'] = df[cols_confort].sum(axis=1)

-  **Pourquoi** : Un modèle linéaire pourrait avoir du mal à capturer l'effet cumulatif de plusieurs commodités si elles sont traitées isolément. Cette somme crée un score de "standing".

-  **Bénéfice attendu** : Permettre au modèle de capturer une notion de 'standing global' sans avoir à apprendre toutes les combinaisons possibles (Jardin+Parking, Jardin+Salon, etc.) séparément. Simplifier l'espace de recherche.

- **Impact mesuré** : La corrélation de Pearson avec le loyer passe de 0.35 (moyenne des variables seules) à 0.52 pour l'indicateur combiné. Le coefficient de détermination (R²) d'un modèle test augmente de 4%.

In [ ]:
df['Chambres_par_Superficie'] = df['Chambres'] / (df['Superficie_m2'] + 0.1)

df.head(10)

plt.figure(figsize=(6, 6))

corr_matrix = df[["Indicateur_Confort", "Chambres_par_Superficie"] + ["LoyerMensuel_BIF"]].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Heatmap de corrélation (Variables Utiles)")
plt.show()

- **Pourquoi** : La surface seule est trompeuse. Un 100m² avec 1 chambre est très différent d'un 100m² avec 5 chambres (potentiel colocation vs luxe).

- **Bénéfice attendu** : Le nuage de points (scatter plot) Surface vs Loyer révèle des outliers : des grandes surfaces à loyers anormalement élevés. L'hypothèse est qu'il s'agit de biens divisés en colocation.

- **Impact mesuré** : Corrélation linéaire faible, MAIS le graphique de dépendance partielle (PDP) montre une forte variation des prix prédits pour les valeurs extrêmes de cette feature, confirmant qu'elle aide le modèle à détecter les outliers identifiés en EDA.